In [ ]:
import numpy as np
import pandas as pd
import synapseclient

# Mapping for goldstandard and submission views by task
GOLDSTANDARD_SYNIDS = {1: "syn68736530", 2: "syn68736533"}
SUBMISSION_VIEWS = {"Task 1": "syn68879001", "Task 2": "syn68878940"}
INDEX_COL = "stimulus"

# Synapse login
syn = synapseclient.Synapse()
syn.login()

In [123]:
def load_goldstandard(syn, task):
    gold_synid = GOLDSTANDARD_SYNIDS[task]
    file_entity = syn.get(gold_synid, downloadFile=True)
    file_path = file_entity.path
    gold_df = pd.read_csv(file_path)
    return gold_df
    
#create try except where you try to get the team name from synapse and if it fails you just get the user
def get_name(id: str|int) -> str:
    try:
        return syn.getTeam(id).name
    except synapseclient.core.exceptions.SynapseHTTPError:
        try:
            return syn.getUserProfile(id).userName
        except synapseclient.core.exceptions.SynapseHTTPError:
            return "NA"
    except ValueError:
        return "NA - id is not correct"

def load_team_predictions(syn, submissions_df):
    team_dfs = []
    # No handling of the submission that's not recorded in the submission table 
    # for task 2 required given the late submission is not in the top 2 or 3
    for _, row in submissions_df.iterrows():
        team_id = row['submitterid']
        sub_id = row['id']
        file_path = syn.getSubmission(sub_id)['filePath']
        df = pd.read_csv(file_path)
        df = df.sort_values(INDEX_COL).reset_index(drop=True)
        feature_cols = [col for col in df.columns if col != INDEX_COL]
        rename_dict = {col: f"team_{team_id}_{col}" for col in feature_cols}
        df = df.rename(columns=rename_dict)
        df = df[[INDEX_COL] + list(rename_dict.values())]
        team_dfs.append(df)
    merged_df = team_dfs[0]
    for df in team_dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=INDEX_COL, how='outer')
    return merged_df

def select_final_round_submissions(
    syn: synapseclient.Synapse, subview_id: str, evaluation_id: str
) -> pd.DataFrame:
    """
    Get final round submissions from synapse tables that include all submissions for both rounds.
    Outputs a final averaged rank and sorted leaderboard.
    Only submissions from the specified evaluation are considered.
    For each submitter, but one team, only the submission closest to August 8, 2025 is kept.
    If any of the IDs [9756929, 9756930, 9756939, 9756938, 9756943, 9756942] are present,
    keep only 9756929 if present, otherwise keep only 9756930 if present, and remove the rest.
    Also manually add the permitted late submission for Task 2.
    """

    query = (
        f"SELECT id, pearson_correlation, cosine, createdOn, submitterid FROM {subview_id} "
        f"WHERE score_status = 'SCORED' "
    )
    submissions = syn.tableQuery(query).asDataFrame()

    # Special handling for the specified IDs
    replace_ids = [9756929, 9756930, 9756939, 9756938, 9756943, 9756942]
    present_ids = [rid for rid in replace_ids if rid in submissions['id'].values]
    if present_ids:
        if 9756929 in present_ids:
            submissions = submissions[~submissions['id'].isin(replace_ids) | (submissions['id'] == 9756929)]
        elif 9756930 in present_ids:
            submissions = submissions[~submissions['id'].isin(replace_ids) | (submissions['id'] == 9756930)]
        else:
            submissions = submissions[~submissions['id'].isin(replace_ids)]
    
    # Keep both team_id (submitterid) and canonical team_name
    submissions['team_id'] = submissions['submitterid'].astype(str)
    submissions['team_name'] = submissions['team_id'].apply(get_name)

    # Find the submission closest to August 8, 2025 for each submitter
    target_date = int(pd.Timestamp("2025-08-08T23:59:59Z").timestamp() * 1000)
    submissions['createdOn_diff'] = np.abs(submissions['createdOn'] - target_date)
    submissions = submissions.sort_values(['submitterid', 'createdOn_diff'])
    submissions = submissions.groupby('submitterid', as_index=False).first()

    # Ranking
    submissions['pearson_rank'] = submissions['pearson_correlation'].rank(
        ascending=False, method="min", na_option='bottom')
    submissions['cosine_rank'] = submissions['cosine'].rank(
        ascending=True, method="min", na_option='bottom')
    submissions['final_rank'] = (
        submissions['pearson_rank'] + submissions['cosine_rank']) / 2

    # Select the top 3 ranked submissions for future bootstrapping
    top_submissions = submissions.nsmallest(3, 'final_rank')

    return submissions, top_submissions


## Task 1 Processing

In [ ]:
subview_id = SUBMISSION_VIEWS[
        f"Task {1}"]
submissions_df = select_final_round_submissions(
        syn, subview_id, "Final Round DREAM Olfactory Mixtures Prediction Challenge 2025 - Task 1")

Downloaded syn68879001 to /Users/mdiaz/.synapseCache/91/164161091/SYNAPSE_TABLE_QUERY_164161091.csv


## View all submissions for Task 1 and their scores and ranks

In [125]:
submissions_df[0]

,submitterid,id,pearson_correlation,cosine,createdOn,team_id,team_name,createdOn_diff,pearson_rank,cosine_rank,final_rank
0,2176686,9756103,0.712790,0.195575,1753902388392,2176686,SuleimanKhan,795210608,8.0,7.0,7.5
1,3319559,9756924,0.719952,0.190646,1754680668286,3319559,yuanfang.guan,16930714,5.0,5.0,5.0
2,3410046,9755761,0.694351,0.209255,1753634625355,3410046,dskhanirfan,1062973645,11.0,13.0,12.0
3,3444251,9756459,0.617820,0.255167,1754357313286,3444251,jgburk,340285714,21.0,21.0,21.0
4,3449866,9756906,0.010114,0.638490,1754661225312,3449866,Metformin-121,36373688,26.0,26.0,26.0
5,3501700,9755325,0.538023,0.329100,1753195170975,3501700,Songxuebo,1502428025,24.0,24.0,24.0
6,3506852,9756908,0.730260,0.179923,1754667145501,3506852,PL21,30453499,2.0,2.0,2.0
7,3516194,9756912,0.752053,0.176111,1754667730603,3516194,nachman.keren,29868397,1.0,1.0,1.0
8,3542349,9755617,0.694188,0.208259,1753450580486,3542349,AyyazAzeem,1247018514,12.0,11.0,11.5
9,3544441,9755381,0.561040,0.292149,1753261269658,3544441,OFI,1436329342,23.0,23.0,23.0


In [126]:
team_predictions_top_t1 = load_team_predictions(syn, submissions_df[1])
team_predictions_all_t1 = load_team_predictions(syn, submissions_df[0])

## Load the submission data for the top 3 teams -- Task 1

In [127]:
team_predictions_top_t1

,stimulus,team_3516194_Green,team_3516194_Cucumber,team_3516194_Herbal,team_3516194_Mint,team_3516194_Woody,team_3516194_Pine,team_3516194_Floral,team_3516194_Powdery,team_3516194_Fruity,...,team_3550368_Phenolic,team_3550368_Animal,team_3550368_Medicinal,team_3550368_Cooling,team_3550368_Sharp,team_3550368_Chlorine,team_3550368_Alcoholic,team_3550368_Plastic,team_3550368_Ozone,team_3550368_Metallic
0,A097,0.163951,0.592068,0.076013,0.136553,0.067981,0.064507,0.108345,0.107931,1.117217,...,0.228182,0.039660,0.376824,0.235058,0.319333,0.102149,0.836267,0.049630,0.110358,0.018073
1,A317,0.127255,0.038375,0.059370,0.009688,0.085294,0.018515,0.112842,0.039311,0.462239,...,0.049953,0.190075,0.001068,0.000000,0.215837,0.022045,0.051667,0.068330,0.027690,0.019519
2,A483,0.785323,0.563811,0.229682,0.035176,0.101651,0.137052,0.139448,0.117391,0.531038,...,0.013584,0.000000,0.026852,0.065758,0.007939,0.123487,0.141093,0.055627,0.236169,0.017535
3,A504,0.382762,0.110396,0.394816,0.331251,0.169675,0.233465,0.306070,0.148835,0.498382,...,0.251922,0.000000,0.394764,0.375218,0.504320,0.311873,0.537410,0.103366,0.262696,0.090098
4,B003,0.275184,0.036155,0.271954,0.069579,0.115879,0.101472,0.085669,0.032859,0.564366,...,0.133704,0.000000,0.179570,0.104150,0.211555,0.049891,0.809801,0.034311,0.135897,0.034626
5,B082,0.166432,0.091333,0.064416,0.111404,0.106331,0.029153,0.143997,0.087178,1.658852,...,0.075241,0.032749,0.220855,0.177993,0.089581,0.051515,0.368505,0.029951,0.059277,0.008832
6,B217,0.153360,0.032688,0.066880,0.054199,0.077503,0.031086,0.123805,0.084038,1.215262,...,0.098705,0.005777,0.097031,0.069195,0.016232,0.031311,0.275065,0.024378,0.053608,0.015528
7,C038,0.211200,0.049361,0.063589,0.013128,0.155566,0.017564,0.016289,0.048605,0.086504,...,0.050899,0.107185,0.101294,0.049808,0.164218,0.117319,0.197508,0.055215,0.002426,0.017050
8,C114,0.092585,0.015121,0.099700,0.029877,0.139629,0.021463,0.230658,0.098934,0.306668,...,0.330983,0.326627,0.082689,0.018046,0.256541,0.084145,0.030957,0.286382,0.011503,0.019729
9,C208,0.162733,0.060024,0.203922,0.087215,0.142459,0.091825,0.075369,0.113933,0.370594,...,0.345301,0.026792,0.532626,0.467507,0.506842,0.122287,1.191625,0.087064,0.069724,0.049671


## Load the submission data for the all teams -- Task 1

In [128]:
team_predictions_all_t1

,stimulus,team_2176686_Green,team_2176686_Cucumber,team_2176686_Herbal,team_2176686_Mint,team_2176686_Woody,team_2176686_Pine,team_2176686_Floral,team_2176686_Powdery,team_2176686_Fruity,...,team_3550943_Phenolic,team_3550943_Animal,team_3550943_Medicinal,team_3550943_Cooling,team_3550943_Sharp,team_3550943_Chlorine,team_3550943_Alcoholic,team_3550943_Plastic,team_3550943_Ozone,team_3550943_Metallic
0,A097,0.137109,0.073554,0.107456,0.139781,0.102030,0.039611,0.066723,0.035725,0.922672,...,0.245494,0.006573,0.262941,0.190191,0.316361,0.085166,0.924170,0.088572,0.082674,0.022632
1,A317,0.109404,0.026696,0.000000,0.000000,0.027609,0.000000,0.056309,0.024320,0.397615,...,0.145820,0.247221,0.062766,0.040021,0.196520,0.068276,0.022823,0.109567,0.015370,0.000000
2,A483,1.038557,1.012992,0.394256,0.072514,0.129511,0.112356,0.228565,0.113536,0.280496,...,0.059237,0.004129,0.187753,0.127951,0.058065,0.134430,0.044859,0.173372,0.323354,0.014739
3,A504,0.394507,0.191976,0.361247,0.143402,0.132808,0.247334,0.418946,0.175509,0.303528,...,0.319257,0.000000,1.131387,0.875749,0.610240,0.282786,0.283425,0.181040,0.371474,0.000578
4,B003,0.237016,0.081480,0.156629,0.052617,0.089477,0.128562,0.095304,0.014450,0.692110,...,0.778656,0.000000,0.843796,0.764753,1.188034,0.299753,2.536840,0.176662,0.160077,0.040130
5,B082,0.150759,0.059169,0.067941,0.092942,0.085546,0.000000,0.092008,0.072343,1.061987,...,0.192512,0.036743,0.171510,0.219227,0.091521,0.063825,0.382617,0.039179,0.086435,0.001696
6,B217,0.105781,0.076973,0.016083,0.057968,0.071498,0.000000,0.059322,0.060508,1.189009,...,0.224213,0.000000,0.272604,0.228793,0.101220,0.092936,0.713700,0.093628,0.064735,0.017739
7,C038,0.260985,0.050158,0.036854,0.039084,0.144994,0.010295,0.000000,0.011679,0.067937,...,0.210913,0.081736,0.286500,0.207271,0.368761,0.198003,0.393523,0.108514,0.067028,0.001046
8,C114,0.122574,0.045364,0.084806,0.024164,0.094732,0.026084,0.048335,0.065738,0.142877,...,0.321759,0.330155,0.090369,0.076909,0.306592,0.086928,0.000000,0.326482,0.045107,0.013390
9,C208,0.249047,0.129258,0.133944,0.089803,0.124975,0.037323,0.014592,0.020504,0.502852,...,0.407195,0.015127,0.664170,0.536653,0.409712,0.188738,1.426635,0.186886,0.077760,0.063468


## Save all prediction data to CSV -- Task 1

In [129]:
team_predictions_all_t1.to_csv("team_predictions_all_t1.csv", index=False)

## Task 2 Processing

In [130]:
subview_id2 = SUBMISSION_VIEWS[
        f"Task {2}"]
submissions_df2 = select_final_round_submissions(
        syn, subview_id2, "Final Round DREAM Olfactory Mixtures Prediction Challenge 2025 - Task 2")

Downloaded syn68878940 to /Users/mdiaz/.synapseCache/93/164161093/SYNAPSE_TABLE_QUERY_164161093.csv


## View all submissions for Task 2 and their scores and ranks

In [131]:
submissions_df2[0]

,submitterid,id,pearson_correlation,cosine,createdOn,team_id,team_name,createdOn_diff,pearson_rank,cosine_rank,final_rank
0,3319559,9756951,0.789684,0.136842,1754695242525,3319559,yuanfang.guan,2356475,2.0,2.0,2.0
1,3444251,9756460,0.717501,0.178520,1754358241536,3444251,jgburk,339357464,9.0,9.0,9.0
2,3449866,9756910,0.743457,0.170847,1754667403131,3449866,Metformin-121,30195869,6.0,8.0,7.0
3,3501700,9755326,0.622921,0.231619,1753195204441,3501700,Songxuebo,1502394559,14.0,14.0,14.0
4,3506852,9756909,0.789749,0.133586,1754667217304,3506852,PL21,30381696,1.0,1.0,1.0
5,3544441,9755465,0.368671,0.367259,1753347764068,3544441,OFI,1349834932,15.0,15.0,15.0
6,3546171,9756421,0.682692,0.200900,1754318701189,3546171,BBKCS2025,378897811,11.0,11.0,11.0
7,3546192,9756916,0.775878,0.142796,1754667973952,3546192,IM-Scent,29625048,4.0,4.0,4.0
8,3546766,9756930,0.736943,0.167478,1754687833068,3546766,SystemsCBLab_OMP,9765932,7.0,6.0,6.5
9,3547318,9756959,0.766696,0.149783,1754697445888,3547318,VSC25,153112,5.0,5.0,5.0


## Load the submission data for the top 3 teams -- Task 2

In [132]:
team_predictions2_top_t2 = load_team_predictions(syn, submissions_df2[1])
team_predictions2_all_t2 = load_team_predictions(syn, submissions_df2[0])


In [133]:
team_predictions2_top_t2

,stimulus,team_3506852_Green,team_3506852_Cucumber,team_3506852_Herbal,team_3506852_Mint,team_3506852_Woody,team_3506852_Pine,team_3506852_Floral,team_3506852_Powdery,team_3506852_Fruity,...,team_3550368_Phenolic,team_3550368_Animal,team_3550368_Medicinal,team_3550368_Cooling,team_3550368_Sharp,team_3550368_Chlorine,team_3550368_Alcoholic,team_3550368_Plastic,team_3550368_Ozone,team_3550368_Metallic
0,AA322,0.212778,0.027949,0.256452,0.058720,0.386482,0.080850,0.272239,0.200554,0.176603,...,0.017855,0.081199,0.045978,0.022544,0.021803,0.009799,0.000000,0.026668,0.000000,0.009012
1,AA374,0.261738,0.035842,0.270668,0.194127,0.175643,0.183624,0.466119,0.231138,0.799090,...,0.066478,0.048864,0.165897,0.170941,0.111467,0.040420,0.089540,0.160298,0.137297,0.040133
2,AA444,0.267285,0.057054,0.256265,0.066583,0.248189,0.073803,0.147573,0.229213,0.524255,...,0.165192,0.044278,0.110873,0.160629,0.162787,0.134241,0.489573,0.120894,0.101768,0.018814
3,AA524,0.113640,0.016293,0.314186,0.129596,0.379520,0.119354,0.220967,0.404764,0.232298,...,0.242093,0.120945,0.167438,0.104429,0.059049,0.069965,0.184147,0.172325,0.218833,0.026094
4,AA616,0.147073,0.023076,0.143403,0.059253,0.158702,0.142605,0.127613,0.439765,0.203251,...,0.124617,0.045697,0.101567,0.086256,0.002775,0.075142,0.086125,0.027788,0.030214,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,AP848,0.423530,0.031043,0.592145,0.232750,0.493351,0.230240,0.426980,0.327129,0.816709,...,0.153523,0.144864,0.193240,0.117152,0.180453,0.133158,0.279904,0.291391,0.187366,0.046217
126,AQ310,0.153784,0.031048,0.380804,0.108024,0.236134,0.113208,0.198479,0.160214,0.488540,...,0.396529,0.408400,0.367808,0.268776,0.232272,0.084115,0.514834,0.246324,0.080888,0.043673
127,AQ810,0.430313,0.149785,0.294652,0.099273,0.356624,0.209907,0.521266,0.140443,0.327297,...,0.275900,0.298208,0.192210,0.170666,0.340747,0.122543,0.285850,0.242653,0.117031,0.182033
128,AQ862,0.358889,0.027695,0.304280,0.108167,0.330465,0.101834,0.131102,0.135755,0.478872,...,0.139742,0.097366,0.199610,0.173428,0.139169,0.038954,0.299123,0.216242,0.031745,0.066303


## Load the submission data for the all teams -- Task 2

In [134]:
team_predictions2_all_t2

,stimulus,team_3319559_Alcoholic,team_3319559_Ammonia,team_3319559_Animal,team_3319559_Berry,team_3319559_BrownSpice,team_3319559_Burnt,team_3319559_Buttery,team_3319559_Caramellic,team_3319559_Cheesy,...,team_3550943_Phenolic,team_3550943_Animal,team_3550943_Medicinal,team_3550943_Cooling,team_3550943_Sharp,team_3550943_Chlorine,team_3550943_Alcoholic,team_3550943_Plastic,team_3550943_Ozone,team_3550943_Metallic
0,AA322,0.039609,0.033792,0.061942,0.080459,0.040246,0.070566,0.583746,0.173101,1.314407,...,0.015566,0.031355,0.000000,0.140649,0.000000,0.031355,0.062709,0.015566,0.015677,0.000000
1,AA374,0.147282,0.081045,0.088755,0.437717,0.051351,0.043556,0.260085,0.103768,0.720514,...,0.248612,0.015613,0.463526,0.283034,0.072259,0.182525,0.129751,0.185283,0.141420,0.022029
2,AA444,0.375761,0.113571,0.068661,0.108942,0.085381,0.064112,0.403897,0.200546,0.125187,...,0.338449,0.000000,0.032178,0.096782,0.032178,0.160974,0.451733,0.016172,0.016172,0.000000
3,AA524,0.204544,0.108047,0.133868,0.081262,0.408334,0.161865,0.077063,0.180725,0.127828,...,0.000000,0.068312,0.221813,0.065021,0.000000,0.047531,0.209875,0.063375,0.031688,0.000000
4,AA616,0.110411,0.034796,0.026552,0.084320,0.061287,0.055929,1.334464,0.233446,0.375847,...,0.244827,0.000000,0.032705,0.112620,0.032705,0.147174,0.212584,0.015891,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,AP848,0.423490,0.234303,0.171663,0.431279,0.379658,0.128451,0.044697,0.392957,0.189408,...,0.276067,0.114650,0.325544,0.161833,0.263615,0.145856,0.571286,0.156203,0.156959,0.037330
126,AQ310,0.465474,0.275059,0.341425,0.219475,0.082102,0.077368,0.107460,0.092612,0.177299,...,0.012618,0.063409,0.012618,0.046371,0.113558,0.000000,0.253637,0.113558,0.063088,0.050470
127,AQ810,0.241932,0.337783,0.227928,0.056150,0.098739,0.092323,0.039486,0.090953,0.048137,...,0.165388,0.396047,0.126882,0.063702,0.218758,0.020271,0.120270,0.080769,0.135255,0.027763
128,AQ862,0.322452,0.196224,0.091698,0.208175,0.139340,0.105350,0.080010,0.362868,0.112293,...,0.377307,0.000000,0.587323,0.071987,0.116979,0.385287,0.371363,0.309622,0.156650,0.000000


## Save all prediction data to CSV -- Task 2

In [135]:
team_predictions2_all_t2.to_csv("team_predictions2_all_t2.csv", index=False)